In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv('train.csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 2 rows:")
df.head(2)

Shape: (100000, 28)

Columns: ['ID', 'Customer_ID', 'Month', 'Name', 'Age', 'SSN', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Type_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Payment_of_Min_Amount', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance', 'Credit_Score']

First 2 rows:


C:\Users\Shreya\AppData\Local\Temp\ipykernel_6772\3839676707.py:4: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('train.csv')


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.82262,22 Years and 1 Months,No,49.574949,80.41529544,High_spent_Small_value_payments,312.4940887,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.94496,NaN,No,49.574949,118.2802216,Low_spent_Large_value_payments,284.6291625,Good


In [5]:
df = df.drop(columns=['ID', 'Customer_ID', 'SSN', 'Name'], errors='ignore')
df = df.drop_duplicates()
df = df.dropna()
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Annual_Income'] = pd.to_numeric(df['Annual_Income'], errors='coerce')
df['Outstanding_Debt'] = pd.to_numeric(df['Outstanding_Debt'], errors='coerce')
df['Monthly_Balance'] = pd.to_numeric(df['Monthly_Balance'], errors='coerce')
df['Amount_invested_monthly'] = pd.to_numeric(df['Amount_invested_monthly'], errors='coerce')
df['Num_of_Delayed_Payment'] = pd.to_numeric(df['Num_of_Delayed_Payment'], errors='coerce')
df['Num_of_Loan'] = pd.to_numeric(df['Num_of_Loan'], errors='coerce')
df['Changed_Credit_Limit'] = pd.to_numeric(df['Changed_Credit_Limit'], errors='coerce')
df['Num_Credit_Inquiries'] = pd.to_numeric(df['Num_Credit_Inquiries'], errors='coerce')
df = df.dropna()
conn = sqlite3.connect('credit_analysis.db')
df.to_sql('credit_score', conn, if_exists='replace', index=False)
print("Data loaded! Total rows:", len(df))

Data loaded! Total rows: 44568


In [6]:
query1 = """
SELECT 
    Credit_Score,
    COUNT(*) AS Total_Customers,
    ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM credit_score), 2) AS Percentage
FROM credit_score
GROUP BY Credit_Score
ORDER BY Total_Customers DESC
"""
result1 = pd.read_sql(query1, conn)
print("=== Credit Score Distribution ===")
print(result1)

=== Credit Score Distribution ===
  Credit_Score  Total_Customers  Percentage
0     Standard            23546       52.83
1         Poor            13870       31.12
2         Good             7152       16.05


In [7]:
query2 = """
SELECT 
    Credit_Score,
    ROUND(AVG(Annual_Income), 0) AS Avg_Annual_Income,
    ROUND(AVG(Monthly_Inhand_Salary), 0) AS Avg_Monthly_Salary,
    ROUND(AVG(Outstanding_Debt), 0) AS Avg_Outstanding_Debt,
    ROUND(AVG(Num_of_Delayed_Payment), 1) AS Avg_Delayed_Payments,
    ROUND(AVG(Credit_Utilization_Ratio), 2) AS Avg_Credit_Utilization,
    ROUND(AVG(Total_EMI_per_month), 0) AS Avg_Monthly_EMI
FROM credit_score
GROUP BY Credit_Score
ORDER BY 
    CASE Credit_Score 
        WHEN 'Good' THEN 1 
        WHEN 'Standard' THEN 2 
        WHEN 'Poor' THEN 3 
    END
"""
result2 = pd.read_sql(query2, conn)
print("=== Average Financial Profile by Credit Score ===")
print(result2)

=== Average Financial Profile by Credit Score ===
  Credit_Score  Avg_Annual_Income  Avg_Monthly_Salary  Avg_Outstanding_Debt  \
0         Good           197063.0              5273.0                 821.0   
1     Standard           184042.0              4096.0                1351.0   
2         Poor           167144.0              3242.0                2138.0   

   Avg_Delayed_Payments  Avg_Credit_Utilization  Avg_Monthly_EMI  
0                  28.0                   32.55           1563.0  
1                  32.5                   32.23           1381.0  
2                  31.9                   31.98           1407.0  


In [8]:
query3 = """
SELECT 
    Occupation,
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN Credit_Score = 'Good' THEN 1 ELSE 0 END) AS Good_Score,
    SUM(CASE WHEN Credit_Score = 'Standard' THEN 1 ELSE 0 END) AS Standard_Score,
    SUM(CASE WHEN Credit_Score = 'Poor' THEN 1 ELSE 0 END) AS Poor_Score,
    ROUND(SUM(CASE WHEN Credit_Score = 'Good' THEN 1 ELSE 0 END)*100.0/COUNT(*), 2) AS Good_Percent
FROM credit_score
GROUP BY Occupation
ORDER BY Good_Percent DESC
"""
result3 = pd.read_sql(query3, conn)
print("=== Credit Score by Occupation ===")
print(result3)

=== Credit Score by Occupation ===
       Occupation  Total_Customers  Good_Score  Standard_Score  Poor_Score  \
0         Teacher             2791         494            1386         911   
1      Journalist             2643         461            1339         843   
2       Architect             2821         484            1503         834   
3        Musician             2631         446            1377         808   
4       Developer             2788         467            1428         893   
5      Accountant             2801         465            1467         869   
6          Doctor             2831         463            1559         809   
7         Manager             2622         427            1393         802   
8       Scientist             2774         451            1447         876   
9   Media_Manager             2759         441            1524         794   
10       Engineer             2776         441            1427         908   
11         Lawyer            

In [9]:
query4 = """
SELECT 
    Credit_Score,
    ROUND(AVG(Delay_from_due_date), 1) AS Avg_Days_Delayed,
    ROUND(AVG(Num_of_Delayed_Payment), 1) AS Avg_Num_Delayed_Payments,
    ROUND(AVG(Outstanding_Debt), 0) AS Avg_Outstanding_Debt
FROM credit_score
GROUP BY Credit_Score
ORDER BY 
    CASE Credit_Score 
        WHEN 'Good' THEN 1 
        WHEN 'Standard' THEN 2 
        WHEN 'Poor' THEN 3 
    END
"""
result4 = pd.read_sql(query4, conn)
print("=== Impact of Delayed Payments on Credit Score ===")
print(result4)

=== Impact of Delayed Payments on Credit Score ===
  Credit_Score  Avg_Days_Delayed  Avg_Num_Delayed_Payments  \
0         Good              11.0                      28.0   
1     Standard              20.4                      32.5   
2         Poor              30.4                      31.9   

   Avg_Outstanding_Debt  
0                 821.0  
1                1351.0  
2                2138.0  


In [10]:
query5 = """
SELECT 
    CASE 
        WHEN Num_Bank_Accounts <= 2 THEN 'Low (1-2 accounts)'
        WHEN Num_Bank_Accounts BETWEEN 3 AND 5 THEN 'Medium (3-5 accounts)'
        ELSE 'High (6+ accounts)'
    END AS Bank_Account_Group,
    COUNT(*) AS Total_Customers,
    ROUND(AVG(Credit_Utilization_Ratio), 2) AS Avg_Credit_Utilization,
    SUM(CASE WHEN Credit_Score = 'Good' THEN 1 ELSE 0 END) AS Good_Score,
    SUM(CASE WHEN Credit_Score = 'Poor' THEN 1 ELSE 0 END) AS Poor_Score,
    ROUND(SUM(CASE WHEN Credit_Score = 'Good' THEN 1 ELSE 0 END)*100.0/COUNT(*), 2) AS Good_Percent
FROM credit_score
GROUP BY Bank_Account_Group
ORDER BY Good_Percent DESC
"""
result5 = pd.read_sql(query5, conn)
print("=== Credit Score by Number of Bank Accounts ===")
print(result5)

=== Credit Score by Number of Bank Accounts ===
      Bank_Account_Group  Total_Customers  Avg_Credit_Utilization  Good_Score  \
0     Low (1-2 accounts)             5272                   32.83        2404   
1  Medium (3-5 accounts)            15400                   32.38        3555   
2     High (6+ accounts)            23896                   31.95        1193   

   Poor_Score  Good_Percent  
0         850         45.60  
1        2554         23.08  
2       10466          4.99  


In [11]:
query6 = """
SELECT 
    Payment_Behaviour,
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN Credit_Score = 'Good' THEN 1 ELSE 0 END) AS Good_Score,
    SUM(CASE WHEN Credit_Score = 'Poor' THEN 1 ELSE 0 END) AS Poor_Score,
    ROUND(SUM(CASE WHEN Credit_Score = 'Good' THEN 1 ELSE 0 END)*100.0/COUNT(*), 2) AS Good_Percent,
    ROUND(SUM(CASE WHEN Credit_Score = 'Poor' THEN 1 ELSE 0 END)*100.0/COUNT(*), 2) AS Poor_Percent
FROM credit_score
GROUP BY Payment_Behaviour
ORDER BY Good_Percent DESC
"""
result6 = pd.read_sql(query6, conn)
print("=== Payment Behaviour vs Credit Score ===")
print(result6)

=== Payment Behaviour vs Credit Score ===
                  Payment_Behaviour  Total_Customers  Good_Score  Poor_Score  \
0   High_spent_Large_value_payments             6069        1297        1412   
1  High_spent_Medium_value_payments             7979        1451        2183   
2   High_spent_Small_value_payments             5109         865        1495   
3    Low_spent_Large_value_payments             4623         781        1390   
4                            !@9#%8             3320         536        1023   
5   Low_spent_Medium_value_payments             6142         969        1992   
6    Low_spent_Small_value_payments            11326        1253        4375   

   Good_Percent  Poor_Percent  
0         21.37         23.27  
1         18.19         27.36  
2         16.93         29.26  
3         16.89         30.07  
4         16.14         30.81  
5         15.78         32.43  
6         11.06         38.63  


In [15]:
query7 = """
SELECT 
    Occupation,
    Annual_Income,
    Outstanding_Debt,
    Num_of_Delayed_Payment,
    Credit_Utilization_Ratio,
    Interest_Rate,
    Credit_Score,
    CASE
        WHEN Num_of_Delayed_Payment > 15 AND Outstanding_Debt > 1000 AND Credit_Utilization_Ratio > 30 THEN 'Very High Risk'
        WHEN Num_of_Delayed_Payment BETWEEN 10 AND 15 AND Outstanding_Debt > 500 THEN 'High Risk'
        WHEN Num_of_Delayed_Payment BETWEEN 5 AND 10 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS Risk_Flag
FROM credit_score
WHERE Credit_Score = 'Poor'
ORDER BY Outstanding_Debt DESC
LIMIT 20
"""
result7 = pd.read_sql(query7, conn)
print("=== High Risk Customer Identification ===")
print(result7)

=== High Risk Customer Identification ===
       Occupation  Annual_Income  Outstanding_Debt  Num_of_Delayed_Payment  \
0   Media_Manager      33343.640           4997.10                    24.0   
1         _______      33343.640           4997.10                    24.0   
2   Media_Manager      33343.640           4997.10                    24.0   
3   Media_Manager      33343.640           4997.10                    24.0   
4        Mechanic       9126.455           4997.05                    24.0   
5         _______       9126.455           4997.05                    26.0   
6        Mechanic       9126.455           4997.05                    22.0   
7        Mechanic       9126.455           4997.05                    24.0   
8        Mechanic       9126.455           4997.05                    24.0   
9        Musician      19213.830           4992.25                    23.0   
10       Musician      19213.830           4992.25                    23.0   
11     Journalist     

In [14]:
query8 = """
SELECT
    COUNT(*) AS Total_Customers,
    COUNT(DISTINCT Occupation) AS Total_Occupations,
    ROUND(AVG(Annual_Income), 0) AS Avg_Annual_Income,
    ROUND(AVG(Outstanding_Debt), 0) AS Avg_Outstanding_Debt,
    ROUND(AVG(Num_of_Delayed_Payment), 1) AS Avg_Delayed_Payments,
    ROUND(AVG(Credit_Utilization_Ratio), 2) AS Avg_Credit_Utilization,
    SUM(CASE WHEN Credit_Score = 'Good' THEN 1 ELSE 0 END) AS Good_Score_Count,
    SUM(CASE WHEN Credit_Score = 'Standard' THEN 1 ELSE 0 END) AS Standard_Score_Count,
    SUM(CASE WHEN Credit_Score = 'Poor' THEN 1 ELSE 0 END) AS Poor_Score_Count
FROM credit_score
"""
result8 = pd.read_sql(query8, conn)
print("=== Executive Summary ===")
print(result8)

=== Executive Summary ===
   Total_Customers  Total_Occupations  Avg_Annual_Income  \
0            44568                 16           180873.0   

   Avg_Outstanding_Debt  Avg_Delayed_Payments  Avg_Credit_Utilization  \
0                1511.0                  31.6                    32.2   

   Good_Score_Count  Standard_Score_Count  Poor_Score_Count  
0              7152                 23546             13870  
